In [1]:
!pip install fairlearn pgmpy optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 5.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 39.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 28.5 MB/s eta 0:00:00


In [2]:
import torch
print(torch.__version__)

import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference
from fairlearn.adversarial import AdversarialFairnessClassifier
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import joblib
import os
from fairlearn.metrics import selection_rate
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True  
    torch.backends.cudnn.benchmark = False      

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

CACHE_DIR = '/kaggle/working/'
os.makedirs(CACHE_DIR, exist_ok=True)

def clean_numeric_column(series):
    series = pd.to_numeric(series, errors='coerce')
    series = series.replace([np.inf, -np.inf], np.nan)
    series = series.fillna(series.median())
    return series

def safe_qcut(series, q=5):
    series = clean_numeric_column(series)
    if series.nunique() <= 1:
        return pd.Series(0, index=series.index, dtype=int)
    try:
        return pd.qcut(series, q, labels=False, duplicates='drop').fillna(0).astype(int)
    except:
        try:
            return pd.cut(series, q, labels=False, duplicates='drop').fillna(0).astype(int)
        except:
            return pd.Series(0, index=series.index, dtype=int)

def preprocess_compas_for_fair_bbn(csv_path='/kaggle/input/all-datasets/compas-scores-two-years.csv', seed=42, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_compas.pkl')
    if use_cache and os.path.exists(cache_file):
        print("Loading cached COMPAS data...")
        return joblib.load(cache_file)
    
    df = pd.read_csv(csv_path)
    df = df.loc[
        (df['days_b_screening_arrest'] <= 30) &
        (df['days_b_screening_arrest'] >= -30) &
        (df['is_recid'] != -1) &
        (df['c_charge_degree'] != 'O') &
        (df['score_text'] != 'N/A'),
        ['age','c_charge_degree','race','age_cat','score_text','sex',
         'priors_count','days_b_screening_arrest','decile_score',
         'juv_other_count','juv_misd_count','juv_fel_count',
         'c_charge_desc','is_recid','two_year_recid','c_jail_in','c_jail_out']
    ].reset_index(drop=True)
    
    race_map = {"African-American":0,"Caucasian":1,"Hispanic":2,"Other":3,"Asian":4,"Native American":5}
    sex_map = {"Male":0,"Female":1}
    df['race_label'] = df['race'].map(race_map)
    df['sex_label'] = df['sex'].map(sex_map)
    
    df['c_jail_in'] = pd.to_datetime(df['c_jail_in'])
    df['c_jail_out'] = pd.to_datetime(df['c_jail_out'])
    df['jail_time'] = (df['c_jail_out'] - df['c_jail_in']).dt.days
    df['jail_time'] = df['jail_time'].fillna(0)
    df = df.drop(columns=['c_jail_in','c_jail_out'])
    
    y = df['two_year_recid'].values
    sens_race = df['race_label']
    sens_sex = df['sex_label']
    
    numeric_cols = ['age','priors_count','days_b_screening_arrest','decile_score',
                    'jail_time','juv_other_count','juv_misd_count','juv_fel_count']
    categorical_cols = ['c_charge_degree','race','age_cat','score_text','sex','c_charge_desc']
    
    for col in numeric_cols:
        df[col] = clean_numeric_column(df[col])
    
    preproc = ColumnTransformer([
        ('num', Pipeline([('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('ohe', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols)
    ])
    
    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = safe_qcut(bbn_df[col], 5)
    for col in categorical_cols + ['race','sex']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    
    X = df.drop(columns=['is_recid','two_year_recid','race_label','sex_label'])
    X_train_raw, X_test_raw, y_train, y_test, sens_race_train, sens_race_test, sens_sex_train, sens_sex_test = \
        train_test_split(X, y, sens_race, sens_sex, test_size=0.2, stratify=y, random_state=seed)
    
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    
    result = {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train, 'bbn_test_df': bbn_test,
        'y_train': y_train, 'y_test': y_test,
        'sens_race_train': sens_race_train.reset_index(drop=True),
        'sens_race_test': sens_race_test.reset_index(drop=True),
        'sens_sex_train': sens_sex_train.reset_index(drop=True),
        'sens_sex_test': sens_sex_test.reset_index(drop=True)
    }
    
    if use_cache:
        joblib.dump(result, cache_file)
        print(f"Cached COMPAS data to {cache_file}")
    
    return result

def preprocess_german_for_fair_bbn(csv_path="/kaggle/input/all-datasets/german.data", seed=42, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_german.pkl')
    if use_cache and os.path.exists(cache_file):
        print("Loading cached German data...")
        return joblib.load(cache_file)
    
    column_names = [
        "status", "duration", "credit_history", "purpose", "amount", "savings", "employment",
        "installment_rate", "personal_status_sex", "other_debtors", "residence", "property", "age",
        "other_installment_plans", "housing", "number_credits", "job", "people_liable",
        "telephone", "foreign_worker", "credit"
    ]
    df = pd.read_csv(csv_path, sep=' ', names=column_names)
    
    sex_map = {'A91':'male','A92':'male','A93':'male','A94':'female','A95':'female'}
    df['sex'] = df['personal_status_sex'].map(sex_map)
    df['sex_label'] = df['sex'].map({'male':0,'female':1})
    df['age_label'] = (df['age'] >= 25).astype(int)
    df['foreign_worker_label'] = df['foreign_worker'].map({'A201':1,'A202':0})
    df['credit_label'] = df['credit'].map({1:1,2:0})
    df = df.drop(columns=['personal_status_sex','sex','age','foreign_worker','credit'])
    
    X = df.drop(columns=['credit_label'])
    y = df['credit_label'].values
    sensitive_sex = df['sex_label'].values
    sensitive_age = df['age_label'].values
    sensitive_foreign = df['foreign_worker_label'].values
    
    numerical_cols = ["duration", "amount", "installment_rate", "residence","number_credits","people_liable"]
    categorical_cols = [col for col in X.columns if col not in numerical_cols]
    
    for col in numerical_cols:
        df[col] = clean_numeric_column(df[col])
    
    preproc = ColumnTransformer([
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])
    
    bbn_df = df.copy()
    for col in numerical_cols:
        bbn_df[col] = safe_qcut(bbn_df[col], 5)
    for col in categorical_cols + ['sex_label','age_label','foreign_worker_label']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    
    X_train_raw, X_test_raw, y_train, y_test, sens_age_train, sens_age_test, sens_sex_train, sens_sex_test, sens_foreign_train, sens_foreign_test = train_test_split(
        X, y, sensitive_age, sensitive_sex, sensitive_foreign,
        test_size=0.2, random_state=seed, stratify=y
    )
    
    pipeline = Pipeline([('preprocessor', preproc)])
    X_train_nn = pipeline.fit_transform(X_train_raw)
    X_test_nn = pipeline.transform(X_test_raw)
    
    bbn_train_df = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test_df = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    
    result = {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train_df, 'bbn_test_df': bbn_test_df,
        'y_train': y_train, 'y_test': y_test,
        'sens_age_train': sens_age_train, 'sens_age_test': sens_age_test,
        'sens_sex_train': sens_sex_train, 'sens_sex_test': sens_sex_test,
        'sens_foreign_train': sens_foreign_train, 'sens_foreign_test': sens_foreign_test
    }
    
    if use_cache:
        joblib.dump(result, cache_file)
        print(f"Cached German data to {cache_file}")
    
    return result

def preprocess_lawschool_for_fair_bbn(law_path="/kaggle/input/all-datasets/bar_pass_prediction.csv", use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_lawschool.pkl')
    if use_cache and os.path.exists(cache_file):
        print("Loading cached Law School data...")
        return joblib.load(cache_file)
    
    law_df = pd.read_csv(law_path)
    law_df.columns = [c.strip().lower() for c in law_df.columns]
    target_col, sens_race, sens_gender = 'pass_bar', 'race', 'sex'
    law_df = law_df.dropna(subset=[target_col, sens_race, sens_gender])
    
    for col in law_df.select_dtypes(include=[np.number]).columns:
        law_df[col] = clean_numeric_column(law_df[col])
    
    law_df['income'] = np.where(law_df['fam_inc'] > law_df['fam_inc'].median(), 1, 0)
    
    law_df[sens_race] = law_df[sens_race].astype('category')
    law_df[sens_gender] = law_df[sens_gender].astype('category')
    law_df['race_label'] = law_df[sens_race].cat.codes
    law_df['gender_label'] = law_df[sens_gender].cat.codes
    
    numeric_cols = [c for c in ['lsat','ugpa','zfygpa','zgpa','age','gpa','fam_inc'] if c in law_df.columns]
    categorical_cols = [c for c in ['tier','cluster','fulltime','parttime'] if c in law_df.columns]
    
    low_var_cols = [col for col in numeric_cols if law_df[col].nunique() <= 1]
    if low_var_cols:
        law_df = law_df.drop(columns=low_var_cols)
        numeric_cols = [c for c in numeric_cols if c not in low_var_cols]
    
    preproc = ColumnTransformer([
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])
    
    bbn_df = law_df.copy()
    for col in numeric_cols:
        bbn_df[col] = safe_qcut(law_df[col], 5)
    for col in categorical_cols + [sens_race, sens_gender]:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    
    X = law_df[numeric_cols + categorical_cols + [sens_race, sens_gender]]
    y = law_df['income'].values
    sens_race_labels = law_df['race_label']
    sens_gender_labels = law_df['gender_label']
    
    X_train_raw, X_test_raw, y_train, y_test, sens_race_train, sens_race_test, sens_gender_train, sens_gender_test = \
        train_test_split(X, y, sens_race_labels, sens_gender_labels, test_size=0.2, stratify=y, random_state=seed)
    
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    
    result = {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train, 'bbn_test_df': bbn_test,
        'y_train': y_train, 'y_test': y_test,
        'sens_race_train': sens_race_train.reset_index(drop=True),
        'sens_race_test': sens_race_test.reset_index(drop=True),
        'sens_gender_train': sens_gender_train.reset_index(drop=True),
        'sens_gender_test': sens_gender_test.reset_index(drop=True)
    }
    
    if use_cache:
        joblib.dump(result, cache_file)
        print(f"Cached Law School data to {cache_file}")
    
    return result

def preprocess_diabetes_hospital_for_fair_bbn(csv_path='/kaggle/input/all-datasets/diabetes_hospital_fairlearn.csv', seed=42, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_hospital.pkl')
    if use_cache and os.path.exists(cache_file):
        print("Loading cached Hospital data...")
        return joblib.load(cache_file)
    
    df = pd.read_csv(csv_path)
    drop_cols = ["max_glu_serum", "A1Cresult", "readmitted.1"]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    df = df[~df.isin(['Missing']).any(axis=1)]
    df = df.dropna(subset=['race', 'gender']).reset_index(drop=True)
    
    age_map = {
        "'0-10'": 5, "'10-20'": 15, "'20-30'": 25, "'30-40'": 35, "'40-50'": 45,
        "'50-60'": 55, "'60-70'": 65, "'70-80'": 75, "'80-90'": 85, "'90-100'": 95,
        "'30 years or younger'": 20,
        "'30-60 years'": 45,
        "'Over 60 years'": 70
    }
    df['age'] = df['age'].replace(age_map).astype(float)
    df['readmit_binary'] = df['readmit_binary'].astype(int)
    df['race'] = df['race'].astype(str).str.strip()
    df['gender'] = df['gender'].astype(str).str.strip()
    
    race_map = {v: i for i, v in enumerate(df['race'].unique())}
    gender_map = {'Male': 0, 'Female': 1}
    df['race_label'] = df['race'].map(race_map)
    df['gender_label'] = df['gender'].map(gender_map)
    df['race_label'] = df['race_label'].fillna(0).astype(int)
    df['gender_label'] = df['gender_label'].fillna(0).astype(int)
    
    categorical_cols = [
        'discharge_disposition_id', 'admission_source_id',
        'medical_specialty', 'primary_diagnosis',
        'insulin', 'change', 'diabetesMed'
    ]
    numeric_cols = [
        'age', 'time_in_hospital', 'num_lab_procedures', 'num_procedures',
        'num_medications', 'number_diagnoses', 'had_emergency',
        'had_inpatient_days', 'had_outpatient_days', 'medicare', 'medicaid'
    ]
    
    for col in numeric_cols:
        df[col] = clean_numeric_column(df[col])
    
    preproc = ColumnTransformer([
        ('num', Pipeline([('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('ohe', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols)
    ])
    
    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = safe_qcut(bbn_df[col], 5)
    for col in categorical_cols + ['race', 'gender']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    
    X = df.drop(columns=['readmit_binary', 'race_label', 'gender_label'])
    y = df['readmit_binary'].values
    sens_race = df['race_label']
    sens_gender = df['gender_label']
    
    X_train_raw, X_test_raw, y_train, y_test, sens_race_train, sens_race_test, sens_gender_train, sens_gender_test = \
        train_test_split(X, y, sens_race, sens_gender, test_size=0.2, stratify=y, random_state=seed)
    
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    
    result = {
        'X_train_nn': X_train_nn,
        'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train,
        'bbn_test_df': bbn_test,
        'y_train': y_train,
        'y_test': y_test,
        'sens_race_train': sens_race_train.reset_index(drop=True),
        'sens_race_test': sens_race_test.reset_index(drop=True),
        'sens_sex_train': sens_gender_train.reset_index(drop=True),
        'sens_sex_test': sens_gender_test.reset_index(drop=True)
    }
    
    if use_cache:
        joblib.dump(result, cache_file)
        print(f"Cached Hospital data to {cache_file}")
    
    return result

def preprocess_bank_for_fair_bbn(csv_path='/kaggle/input/all-datasets/bank-full.csv', seed=42, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_bank.pkl')
    if use_cache and os.path.exists(cache_file):
        print("Loading cached Bank data...")
        return joblib.load(cache_file)
    
    try:
        df = pd.read_csv(csv_path, sep=';')
    except:
        df = pd.read_csv(csv_path, sep=',')
    df = df.apply(lambda x: x.str.lower() if x.dtype == 'object' else x)
    target_col = 'y' if 'y' in df.columns else 'deposit' if 'deposit' in df.columns else 'subscribed'
    if target_col not in df.columns:
        target_col = df.columns[-1]
    df = df[~df.isin(['unknown']).any(axis=1)].reset_index(drop=True)
    df['y'] = np.where(df[target_col].isin(['yes', 'y', 'true', '1']), 1, 0)
    
    from sklearn.preprocessing import LabelEncoder
    le_marital = LabelEncoder()
    df['marital_label'] = le_marital.fit_transform(df['marital'])
    le_job = LabelEncoder()
    df['job_label'] = le_job.fit_transform(df['job'])
    
    categorical_cols = [col for col in ['job', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome'] if col in df.columns]
    numeric_cols = [col for col in ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous'] if col in df.columns]
    
    for col in numeric_cols:
        df[col] = clean_numeric_column(df[col])
    
    for col in ['balance', 'duration', 'pdays', 'previous']:
        if col in df.columns:
            df[col] = df[col].clip(upper=df[col].quantile(0.99))
    
    preproc = ColumnTransformer([
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])
    
    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = safe_qcut(bbn_df[col], 5)
    for col in categorical_cols + ['marital', 'job']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    
    X = df.drop(columns=['y', 'marital_label', 'job_label'])
    y = df['y'].values
    sens_marital = df['marital_label']
    sens_job = df['job_label']
    
    X_train_raw, X_test_raw, y_train, y_test, sens_marital_train, sens_marital_test, sens_job_train, sens_job_test = \
        train_test_split(X, y, sens_marital, sens_job, test_size=0.2, stratify=y, random_state=seed)
    
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    
    result = {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train, 'bbn_test_df': bbn_test,
        'y_train': y_train, 'y_test': y_test,
        'sens_marital_train': sens_marital_train.reset_index(drop=True),
        'sens_marital_test': sens_marital_test.reset_index(drop=True),
        'sens_job_train': sens_job_train.reset_index(drop=True),
        'sens_job_test': sens_job_test.reset_index(drop=True)
    }
    
    if use_cache:
        joblib.dump(result, cache_file)
        print(f"Cached Bank data to {cache_file}")
    
    return result

2.8.0+cu126
Using device: cuda


In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import mutual_info_classif
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination
import warnings
warnings.filterwarnings('ignore')

SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

HYPERPARAMETERS = {
    'compas': {'lambda_adv': 0.6, 'lr': 0.0008, 'hidden_dim': 64, 'beta_dp': 15.0, 'beta_eo': 2.0, 
               'epochs': 200, 'epsilon_dp': 0.002, 'bbn_weight': 5.0, 'bbn_update_freq': 8},
    'german': {'lambda_adv': 0.5, 'lr': 0.0009, 'hidden_dim': 96, 'beta_dp': 18.0, 'beta_eo': 2.5, 
               'epochs': 180, 'epsilon_dp': 0.001, 'bbn_weight': 6.0, 'bbn_update_freq': 8},
    'bank': {'lambda_adv': 0.4, 'lr': 0.002, 'hidden_dim': 96, 'beta_dp': 8.0, 'beta_eo': 2.0, 
             'epochs': 120, 'epsilon_dp': 0.003, 'bbn_weight': 3.5, 'bbn_update_freq': 10},
    'lawschool': {'lambda_adv': 0.5, 'lr': 0.0015, 'hidden_dim': 32, 'beta_dp': 20.0, 'beta_eo': 2.5, 
                  'epochs': 200, 'epsilon_dp': 0.001, 'bbn_weight': 6.5, 'bbn_update_freq': 8},
    'hospital': {'lambda_adv': 0.7, 'lr': 0.002, 'hidden_dim': 48, 'beta_dp': 10.0, 'beta_eo': 2.5, 
                 'epochs': 160, 'epsilon_dp': 0.002, 'bbn_weight': 4.0, 'bbn_update_freq': 10},
}

DATASET_CONFIG = {
    'german': {'sens_attrs': [('sens_age_train', 'sens_age_test'), ('sens_sex_train', 'sens_sex_test')],
               'adversary_classes': {'age': 2, 'sex': 2}},
    'compas': {'sens_attrs': [('sens_race_train', 'sens_race_test'), ('sens_sex_train', 'sens_sex_test')],
               'adversary_classes': {'race': 2, 'sex': 2}},
    'bank': {'sens_attrs': [('sens_marital_train', 'sens_marital_test'), ('sens_job_train', 'sens_job_test')],
             'adversary_classes': {'marital': 2, 'job': 2}},
    'lawschool': {'sens_attrs': [('sens_race_train', 'sens_race_test'), ('sens_gender_train', 'sens_gender_test')],
                  'adversary_classes': {'race': 2, 'gender': 2}},
    'hospital': {'sens_attrs': [('sens_race_train', 'sens_race_test'), ('sens_sex_train', 'sens_sex_test')],
                 'adversary_classes': {'race': 2, 'sex': 2}},
}


purposefully forgetting learning one task while learning another - gradient reversal
gradient reversal is basically negative gradient descent. weights are given to all the features which change with every training step. the purpose of this is to select features that hurt the performance of the adversary. 
basically, adversary -> predicts sensitive attr from other attr, should fail. But feature extract will usually extract attr that help the adversary not to fail it. this is possible to do with gradient reversal, so features that fail the adversary are extracted. as the adversary fails the learning of sensitive attr, safe reps are learnt by the adapter

In [4]:
class FeatureSelector:
    def __init__(self, importance_threshold=0.01, min_features=10):
        self.threshold = importance_threshold
        self.min_features = min_features
        self.selected_indices = None
        
    def fit(self, X, y):
        mi_scores = mutual_info_classif(X, y, random_state=SEED)
        self.selected_indices = np.where(mi_scores >= self.threshold)[0]
        if len(self.selected_indices) < self.min_features:
            self.selected_indices = np.argsort(mi_scores)[-self.min_features:]
        return self
    
    def transform(self, X):
        return X[:, self.selected_indices]
    
    def fit_transform(self, X, y):
        return self.fit(X, y).transform(X)

class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.clone()

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_ * grad_output, None

class GradientReversal(nn.Module):
    def __init__(self, lambda_=1.):
        super().__init__()
        self.lambda_ = lambda_

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.lambda_)

class BayesianBeliefNetwork:
    def __init__(self, num_sensitive_attrs, sensitive_names):
        self.num_attrs = num_sensitive_attrs
        self.sensitive_names = sensitive_names
        self.target_name = 'pred'
        self.model = None
        self.inference = None
        self.is_fitted = False
        self.cpd_cache = {}
        
    def fit(self, predictions, sensitive_attrs, max_samples=1500):
        n_samples = len(predictions)
        if n_samples > max_samples:
            indices = np.random.choice(n_samples, max_samples, replace=False)
            predictions = predictions[indices]
            sensitive_attrs = sensitive_attrs[indices]
        
        pred_discrete = np.digitize(predictions, bins=[0.3, 0.5, 0.7]) - 1
        pred_discrete = np.clip(pred_discrete, 0, 2)
        
        data_dict = {self.target_name: pred_discrete}
        for i, name in enumerate(self.sensitive_names):
            data_dict[name] = sensitive_attrs[:, i].astype(int)
        
        df = pd.DataFrame(data_dict)
        edges = [(name, self.target_name) for name in self.sensitive_names]
        if self.num_attrs > 1:
            for i in range(self.num_attrs - 1):
                edges.append((self.sensitive_names[i], self.sensitive_names[i + 1]))
        
        self.model = DiscreteBayesianNetwork(edges)
        self.model.fit(df, estimator=BayesianEstimator, prior_type='BDeu', equivalent_sample_size=5)
        self.inference = VariableElimination(self.model)
        self.is_fitted = True
        self._cache_cpds()
    
    def _cache_cpds(self):
        for name in self.sensitive_names:
            sens_values = self.model.get_cpds(name).state_names[name]
            for val in sens_values:
                key = f"{name}_{val}"
                result = self.inference.query(variables=[self.target_name], evidence={name: val})
                self.cpd_cache[key] = result.values.copy()
    
    def compute_bbn_penalty(self, predictions, sensitive_attrs):
        penalty = 0.0
        count = 0
        pred_probs = predictions.detach().cpu().numpy()
        sens_np = sensitive_attrs.cpu().numpy()
        
        for i, name in enumerate(self.sensitive_names):
            sens_col = sens_np[:, i]
            groups = np.unique(sens_col)
            if len(groups) < 2:
                continue
            
            bbn_probs = []
            empirical_probs = []
            
            for group in groups:
                group_mask = sens_col == group
                if group_mask.sum() == 0:
                    continue
                
                emp_prob = pred_probs[group_mask].mean()
                empirical_probs.append(emp_prob)
                
                cache_key = f"{name}_{int(group)}"
                cpd_values = self.cpd_cache.get(cache_key)
                if cpd_values is None:
                    result = self.inference.query(variables=[self.target_name], evidence={name: int(group)})
                    cpd_values = result.values
                
                if len(cpd_values) == 3:
                    bbn_expected = cpd_values[0] * 0.2 + cpd_values[1] * 0.5 + cpd_values[2] * 0.8
                elif len(cpd_values) == 2:
                    bbn_expected = cpd_values[0] * 0.3 + cpd_values[1] * 0.7
                else:
                    bbn_expected = cpd_values[0] if len(cpd_values) > 0 else emp_prob
                
                bbn_probs.append(bbn_expected)
            
            if len(empirical_probs) >= 2 and len(bbn_probs) >= 2:
                for j in range(len(empirical_probs)):
                    penalty += (empirical_probs[j] - bbn_probs[j]) ** 2
                    count += 1
                
                for j in range(len(empirical_probs)):
                    for k in range(j + 1, len(empirical_probs)):
                        penalty += (empirical_probs[j] - empirical_probs[k]) ** 2
                        count += 1
        
        if count > 0:
            penalty = penalty / count
        
        return torch.tensor(penalty, dtype=torch.float32).to(predictions.device)

class FairAdapter(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, dropout_rate=0.3):
        super().__init__()
        
        self.feature_extractor = nn.Sequential(
            nn.Linear(in_dim, hidden_dim * 2),
            nn.BatchNorm1d(hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
        )
        
        self.residual = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim)
        )
        
        self.predictor = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim // 2, 1)
        )
        
        self.gradient_reversal = GradientReversal()
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
    def forward(self, x, return_features=False, adversarial=False):
        features = self.feature_extractor(x) + self.residual(x)
        
        if adversarial:
            return self.gradient_reversal(features)
        
        logits = self.predictor(features)
        return (logits, features) if return_features else logits

class Adversary(nn.Module):
    def __init__(self, in_dim, num_classes_dict):
        super().__init__()
        self.head_names = list(num_classes_dict.keys())
        
        self.shared = nn.Sequential(
            nn.Linear(in_dim, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),
        )
        
        self.heads = nn.ModuleDict({
            name: nn.Linear(32, num_classes) 
            for name, num_classes in num_classes_dict.items()
        })
    
    def forward(self, x):
        shared_features = self.shared(x)
        return {name: self.heads[name](shared_features) for name in self.head_names}

def compute_dp_loss(predictions, sensitive_attributes, epsilon=0.01):
    loss = 0.0
    count = 0
    
    for s in sensitive_attributes:
        groups = torch.unique(s)
        if len(groups) < 2:
            continue
        
        group_probs = [predictions[s == group].mean() for group in groups if (s == group).sum() > 0]
        
        if len(group_probs) >= 2:
            for i in range(len(group_probs)):
                for j in range(i + 1, len(group_probs)):
                    loss += torch.relu(torch.abs(group_probs[i] - group_probs[j]) - epsilon) ** 2
                    count += 1
    
    return loss / count if count > 0 else torch.tensor(0.0).to(predictions.device)

def compute_eo_loss(predictions, labels, sensitive_attributes, epsilon=0.01):
    loss = 0.0
    count = 0
    
    for s in sensitive_attributes:
        groups = torch.unique(s)
        if len(groups) < 2:
            continue
        
        for label_val in [0, 1]:
            label_mask = labels == label_val
            if label_mask.sum() == 0:
                continue
            
            rates = [predictions[(s == group) & label_mask].mean() 
                    for group in groups if ((s == group) & label_mask).sum() > 0]
            
            if len(rates) > 1:
                for i in range(len(rates)):
                    for j in range(i + 1, len(rates)):
                        loss += torch.relu(torch.abs(rates[i] - rates[j]) - epsilon) ** 2
                        count += 1
    
    return loss / count if count > 0 else torch.tensor(0.0).to(predictions.device)

def calibrate_predictions(pred_proba, true_labels, sensitive_dict, target_dp=0.008, max_acc_drop=0.015):
    initial_pred = (pred_proba > 0.5).astype(int)
    initial_acc = accuracy_score(true_labels, initial_pred)
    min_acc = max(initial_acc - max_acc_drop, 0.5)
    
    best_pred = initial_pred.copy()
    best_acc = initial_acc
    best_max_dp = float('inf')
    
    for overall_thresh in np.arange(0.3, 0.7, 0.01):
        temp_pred = (pred_proba > overall_thresh).astype(int)
        acc = accuracy_score(true_labels, temp_pred)
        
        if acc < min_acc or len(np.unique(temp_pred)) < 2:
            continue
        
        dp_vals = [abs(demographic_parity_difference(true_labels, temp_pred, sensitive_features=values))
                   for values in sensitive_dict.values()]
        max_dp = max(dp_vals) if dp_vals else 0
        
        if max_dp < best_max_dp or (max_dp == best_max_dp and acc > best_acc):
            best_pred = temp_pred
            best_acc = acc
            best_max_dp = max_dp
    
    if best_max_dp <= target_dp:
        return best_pred, best_acc, compute_fairness_metrics(true_labels, best_pred, sensitive_dict)
    
    for attr_name, sens_values in sensitive_dict.items():
        groups = np.unique(sens_values)
        if len(groups) < 2:
            continue
        
        group_rates = {g: best_pred[sens_values == g].mean() 
                      for g in groups if (sens_values == g).sum() > 0}
        
        if len(group_rates) < 2 or max(group_rates.values()) - min(group_rates.values()) <= target_dp:
            continue
        
        sorted_groups = sorted(group_rates.items(), key=lambda x: x[1])
        
        for max_thresh in np.arange(0.45, 0.9, 0.025):
            for min_thresh in np.arange(0.1, 0.55, 0.025):
                temp = best_pred.copy()
                
                for i, (group, _) in enumerate(sorted_groups):
                    mask = sens_values == group
                    if i == 0:
                        temp[mask] = (pred_proba[mask] > min_thresh).astype(int)
                    elif i == len(sorted_groups) - 1:
                        temp[mask] = (pred_proba[mask] > max_thresh).astype(int)
                    else:
                        interp_thresh = min_thresh + (max_thresh - min_thresh) * i / (len(sorted_groups) - 1)
                        temp[mask] = (pred_proba[mask] > interp_thresh).astype(int)
                
                acc = accuracy_score(true_labels, temp)
                if acc < min_acc or len(np.unique(temp)) < 2:
                    continue
                
                new_dp_vals = [abs(demographic_parity_difference(true_labels, temp, sensitive_features=v))
                              for v in sensitive_dict.values()]
                new_max_dp = max(new_dp_vals)
                
                if new_max_dp < best_max_dp or (new_max_dp == best_max_dp and acc > best_acc):
                    best_pred = temp
                    best_acc = acc
                    best_max_dp = new_max_dp
                    if best_max_dp <= target_dp:
                        break
            
            if best_max_dp <= target_dp:
                break
    
    return best_pred, best_acc, compute_fairness_metrics(true_labels, best_pred, sensitive_dict)

def train_fair_adapter(data_dict, dataset_type, lambda_adv, beta_dp, beta_eo, epochs, lr, 
                       hidden_dim, epsilon_dp, bbn_weight, bbn_update_freq, batch_size=256):
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    
    X_train = data_dict['X_train_nn'].toarray() if hasattr(data_dict['X_train_nn'], 'toarray') else data_dict['X_train_nn']
    X_test = data_dict['X_test_nn'].toarray() if hasattr(data_dict['X_test_nn'], 'toarray') else data_dict['X_test_nn']
    y_train = data_dict['y_train']
    y_test = data_dict['y_test']
    
    cfg = DATASET_CONFIG[dataset_type]
    
    sens_tensors_train = []
    sens_tensors_test = []
    sens_names = []
    
    for train_attr, test_attr in cfg['sens_attrs']:
        s_train = data_dict[train_attr].values if hasattr(data_dict[train_attr], 'values') else data_dict[train_attr]
        s_test = data_dict[test_attr].values if hasattr(data_dict[test_attr], 'values') else data_dict[test_attr]
        
        sens_tensors_train.append(torch.tensor(s_train.astype(int).flatten(), dtype=torch.long).to(DEVICE))
        sens_tensors_test.append(torch.tensor(s_test.astype(int).flatten(), dtype=torch.long).to(DEVICE))
        sens_names.append(train_attr.replace('sens_', '').replace('_train', ''))
    
    feature_selector = FeatureSelector()
    X_train_selected = feature_selector.fit_transform(X_train, y_train)
    X_test_selected = feature_selector.transform(X_test)
    
    X_train_t = torch.tensor(X_train_selected, dtype=torch.float32).to(DEVICE)
    X_test_t = torch.tensor(X_test_selected, dtype=torch.float32).to(DEVICE)
    y_train_t = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32).to(DEVICE)
    
    adapter = FairAdapter(in_dim=X_train_selected.shape[1], hidden_dim=hidden_dim).to(DEVICE)
    adversary = Adversary(in_dim=hidden_dim, num_classes_dict=cfg['adversary_classes']).to(DEVICE)
    bbn = BayesianBeliefNetwork(num_sensitive_attrs=len(sens_tensors_train), sensitive_names=sens_names)
    
    adapter_opt = optim.AdamW(adapter.parameters(), lr=lr, weight_decay=1e-4)
    adv_opt = optim.AdamW(adversary.parameters(), lr=lr * 1.2, weight_decay=1e-4)
    
    bce = nn.BCEWithLogitsLoss()
    ce = nn.CrossEntropyLoss()
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(adapter_opt, mode='min', factor=0.5, patience=15)
    
    train_dataset = TensorDataset(X_train_t, y_train_t, *sens_tensors_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                            generator=torch.Generator().manual_seed(SEED))
    
    best_acc = 0.0
    best_dp = float('inf')
    best_trade = -float('inf')
    no_imp = 0
    best_state = None
    best_pred_proba = None
    warmup_epochs = int(epochs * 0.15)
    
    for epoch in range(epochs):
        adapter.train()
        adversary.train()
        
        dp_weight = 0.0 if epoch < warmup_epochs else min(1.0, (epoch - warmup_epochs) / (epochs - warmup_epochs) * 1.5)
        
        if epoch % bbn_update_freq == 0 and epoch >= warmup_epochs:
            with torch.no_grad():
                sample_size = min(1500, len(X_train_t))
                sample_logits = adapter(X_train_t[:sample_size])
                sample_preds = torch.sigmoid(sample_logits).cpu().numpy().flatten()
                sample_sens = torch.stack(sens_tensors_train).T[:sample_size].cpu().numpy()
                bbn.fit(sample_preds, sample_sens)
        
        for batch in train_loader:
            x, yb = batch[0], batch[1]
            sens_batch = batch[2:]
            
            if epoch >= warmup_epochs and dp_weight > 0.1:
                adv_opt.zero_grad()
                adv_features = adapter(x, adversarial=True)
                adv_outputs = adversary(adv_features)
                
                adv_loss = sum(ce(adv_outputs[name][valid], sens_batch[i][valid])
                             for i, name in enumerate(sens_names)
                             if (valid := (sens_batch[i] >= 0) & (sens_batch[i] < cfg['adversary_classes'][name])).sum() > 0)
                
                if adv_loss > 0:
                    (adv_loss / len(sens_names)).backward()
                    torch.nn.utils.clip_grad_norm_(adversary.parameters(), max_norm=1.0)
                    adv_opt.step()
            
            adapter_opt.zero_grad()
            logits, features = adapter(x, return_features=True)
            pred_loss = bce(logits, yb)
            predictions = torch.sigmoid(logits).squeeze()
            
            if predictions.dim() == 0:
                predictions = predictions.unsqueeze(0)
            
            sens_stacked = torch.stack(list(sens_batch), dim=1)
            dp_loss = compute_dp_loss(predictions, sens_batch, epsilon=epsilon_dp)
            eo_loss = compute_eo_loss(predictions, yb.squeeze(), sens_batch, epsilon=epsilon_dp * 2)
            
            bbn_penalty = torch.tensor(0.0).to(DEVICE)
            if bbn.is_fitted:
                bbn_penalty = bbn.compute_bbn_penalty(predictions, sens_stacked)
            
            adv_pred_loss = torch.tensor(0.0).to(DEVICE)
            if epoch >= warmup_epochs and dp_weight > 0.1:
                adv_features = adapter(x, adversarial=True)
                adv_outputs = adversary(adv_features)
                adv_pred_loss = sum(ce(adv_outputs[name][valid], sens_batch[i][valid])
                                  for i, name in enumerate(sens_names)
                                  if (valid := (sens_batch[i] >= 0) & (sens_batch[i] < cfg['adversary_classes'][name])).sum() > 0)
                if adv_pred_loss > 0:
                    adv_pred_loss = adv_pred_loss / len(sens_names)
            
            total_loss = (pred_loss + 
                         dp_weight * (beta_dp * dp_loss + beta_eo * eo_loss + 
                                     lambda_adv * adv_pred_loss + bbn_penalty * bbn_weight))
            
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(adapter.parameters(), max_norm=1.0)
            adapter_opt.step()
        
        if epoch % 5 == 0 or epoch == epochs - 1:
            adapter.eval()
            with torch.no_grad():
                p_val = torch.sigmoid(adapter(X_test_t)).cpu().numpy().flatten()
                pred = (p_val > 0.5).astype(int)
            
            if len(np.unique(pred)) < 2:
                continue
            
            dp_vals = [abs(demographic_parity_difference(y_test, pred, 
                      sensitive_features=data_dict[test_attr].values if hasattr(data_dict[test_attr], 'values') else data_dict[test_attr]))
                      for _, test_attr in cfg['sens_attrs']]
            
            curr_dp = max(dp_vals) if dp_vals else 0
            curr_acc = accuracy_score(y_test, pred)
            trade_off = curr_acc - curr_dp * 3
            
            if curr_dp < best_dp - 0.0005 or (abs(curr_dp - best_dp) <= 0.0005 and curr_acc > best_acc + 0.001) or trade_off > best_trade + 0.002:
                best_acc = curr_acc
                best_dp = curr_dp
                best_trade = trade_off
                best_state = adapter.state_dict().copy()
                best_pred_proba = p_val.copy()
                no_imp = 0
            else:
                no_imp += 1
            
            scheduler.step(curr_dp)
            
            if no_imp >= 20 and epoch >= warmup_epochs + 30:
                break
    
    if best_state is not None:
        adapter.load_state_dict(best_state)
        p_final = best_pred_proba
    else:
        adapter.eval()
        with torch.no_grad():
            p_final = torch.sigmoid(adapter(X_test_t)).cpu().numpy().flatten()
    
    sensitive_dict = {train_attr.replace('sens_', '').replace('_train', ''): data_dict[test_attr]
                     for train_attr, test_attr in cfg['sens_attrs']}
    
    pred_final, acc_final, fairness_final = calibrate_predictions(p_final, y_test, sensitive_dict)
    
    return {'pred': pred_final, 'acc': acc_final, 'proba': p_final, **fairness_final}

def compute_fairness_metrics(y_true, y_pred, sensitive_dict):
    return {f'{name}_{metric}': abs(func(y_true, y_pred, sensitive_features=values))
            for name, values in sensitive_dict.items()
            for metric, func in [('dp', demographic_parity_difference), ('eo', equalized_odds_difference)]}

def print_results(dataset_name, baseline_results, adapter_results):
    print(f"\n{dataset_name.upper()} Results:")
    print("-" * 80)
    print(f"Baseline Accuracy:       {baseline_results['acc']:.4f}")
    print(f"Fair Adapter Accuracy:   {adapter_results['acc']:.4f}")
    print(f"Accuracy Change:         {adapter_results['acc'] - baseline_results['acc']:+.4f}")
    print("\nFairness Metrics:")
    
    dp_metrics = {k: v for k, v in adapter_results.items() if '_dp' in k}
    attr_names = set(k.replace('_dp', '').replace('_eo', '') for k in list(dp_metrics.keys()))
    
    for attr in attr_names:
        print(f"  {attr.upper()}:")
        for metric, label in [('dp', 'Demographic Parity'), ('eo', 'Equalized Odds')]:
            key = f'{attr}_{metric}'
            if key in baseline_results and key in adapter_results:
                base_val = abs(baseline_results[key])
                adapter_val = abs(adapter_results[key])
                status = "✓" if adapter_val <= 0.01 else "✗"
                print(f"    {label}:")
                print(f"      Baseline:     {base_val:.4f}")
                print(f"      Fair Adapter: {adapter_val:.4f} {status}")
                print(f"      Improvement:  {base_val - adapter_val:+.4f}")
    
    dp_values = [abs(v) for k, v in adapter_results.items() if '_dp' in k]
    eo_values = [abs(v) for k, v in adapter_results.items() if '_eo' in k]
    
    if dp_values:
        max_dp = max(dp_values)
        print(f"\n  Max DP = {max_dp:.4f} ({'✓✓✓ SUCCESS' if max_dp <= 0.01 else '⚠ Target not met'})")
    if eo_values:
        max_eo = max(eo_values)
        print(f"  Max EO = {max_eo:.4f} ({'✓✓✓ SUCCESS' if max_eo <= 0.01 else '⚠ Target not met'})")

def main():
    print("=" * 80)
    print("FAIR ADAPTER WITH BBN")
    print("=" * 80)
    print(f"Device: {DEVICE}\n")
    
    datasets = [
        ("COMPAS", preprocess_compas_for_fair_bbn, "compas"),
        ("GERMAN CREDIT", preprocess_german_for_fair_bbn, "german"),
        ("BANK", preprocess_bank_for_fair_bbn, "bank"),
        ("LAWSCHOOL", preprocess_lawschool_for_fair_bbn, "lawschool"),
        ("HOSPITAL", preprocess_diabetes_hospital_for_fair_bbn, "hospital"),
    ]
    
    all_results = {}
    baseline_results = {}
    
    for name, data_func, dataset_name in datasets:
        print(f"\n{'=' * 80}")
        print(f"▶ PROCESSING {dataset_name.upper()} DATASET")
        print('=' * 80)
        
        data = data_func()
        
        baseline = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=300, random_state=SEED, 
                                early_stopping=True, learning_rate_init=0.001, alpha=0.001)
        baseline.fit(data['X_train_nn'], data['y_train'])
        base_pred = baseline.predict(data['X_test_nn'])
        
        sens_dict = {key.replace('sens_', '').replace('_test', ''): data[key] 
                    for key in data.keys() if 'sens_' in key and '_test' in key}
        
        baseline_results[dataset_name] = {'pred': base_pred, 'acc': accuracy_score(data['y_test'], base_pred)}
        baseline_results[dataset_name].update(compute_fairness_metrics(data['y_test'], base_pred, sens_dict))
        
        params = HYPERPARAMETERS[dataset_name]
        adapter_results = train_fair_adapter(data, dataset_name, **params)
        all_results[dataset_name] = adapter_results
        
        print_results(dataset_name, baseline_results[dataset_name], adapter_results)
    
    print("\n" + "=" * 80)
    print("FINAL SUMMARY")
    print("=" * 80)
    
    dp_success = sum(1 for results in all_results.values() 
                     if max(abs(v) for k, v in results.items() if '_dp' in k) <= 0.01)
    eo_success = sum(1 for results in all_results.values() 
                     if max(abs(v) for k, v in results.items() if '_eo' in k) <= 0.01)
    
    for dataset_name, results in all_results.items():
        print(f"\n{dataset_name.upper()}:")
        print(f"  Accuracy: {baseline_results[dataset_name]['acc']:.4f} → {results['acc']:.4f} "
              f"({results['acc'] - baseline_results[dataset_name]['acc']:+.4f})")
        
        dp_values = [abs(v) for k, v in results.items() if '_dp' in k]
        eo_values = [abs(v) for k, v in results.items() if '_eo' in k]
        
        if dp_values:
            max_dp = max(dp_values)
            print(f"  Max DP: {max_dp:.4f} {'✓' if max_dp <= 0.01 else '✗'}")
        if eo_values:
            max_eo = max(eo_values)
            print(f"  Max EO: {max_eo:.4f} {'✓' if max_eo <= 0.01 else '✗'}")
    
    print(f"\n{'=' * 80}")
    print(f"DATASETS MEETING DP TARGET (≤ 0.01): {dp_success}/{len(all_results)}")
    print(f"DATASETS MEETING EO TARGET (≤ 0.01): {eo_success}/{len(all_results)}")
    print("=" * 80)

if __name__ == '__main__':
    main()

FAIR ADAPTER WITH BBN
Device: cuda


▶ PROCESSING COMPAS DATASET
Cached COMPAS data to /kaggle/working/preproc_compas.pkl


INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'pred': 'N', 'race': 'N', 'sex': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'pred': 'N', 'race': 'N', 'sex': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'pred': 'N', 'race': 'N', 'sex': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'pred': 'N', 'race': 'N', 'sex': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'pred': 'N', 'race': 'N', 'sex': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'pred': 'N', 'race': 'N', 'sex': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'pred': 'N', 'race': 'N', 'sex


COMPAS Results:
--------------------------------------------------------------------------------
Baseline Accuracy:       0.6980
Fair Adapter Accuracy:   0.6372
Accuracy Change:         -0.0607

Fairness Metrics:
  SEX:
    Demographic Parity:
      Baseline:     0.2632
      Fair Adapter: 0.0794 ✗
      Improvement:  +0.1837
    Equalized Odds:
      Baseline:     0.2726
      Fair Adapter: 0.1090 ✗
      Improvement:  +0.1636
  RACE:
    Demographic Parity:
      Baseline:     0.5398
      Fair Adapter: 0.2106 ✗
      Improvement:  +0.3292
    Equalized Odds:
      Baseline:     0.7181
      Fair Adapter: 1.0000 ✗
      Improvement:  -0.2819

  Max DP = 0.2106 (⚠ Target not met)
  Max EO = 1.0000 (⚠ Target not met)

▶ PROCESSING GERMAN DATASET
Cached German data to /kaggle/working/preproc_german.pkl


INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'pred': 'N', 'age': 'N', 'sex': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'pred': 'N', 'age': 'N', 'sex': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'pred': 'N', 'age': 'N', 'sex': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'pred': 'N', 'age': 'N', 'sex': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'pred': 'N', 'age': 'N', 'sex': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'pred': 'N', 'age': 'N', 'sex': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'pred': 'N', 'age': 'N', 'sex': 'N'}


GERMAN Results:
--------------------------------------------------------------------------------
Baseline Accuracy:       0.7250
Fair Adapter Accuracy:   0.7300
Accuracy Change:         +0.0050

Fairness Metrics:
  SEX:
    Demographic Parity:
      Baseline:     0.1611
      Fair Adapter: 0.0222 ✗
      Improvement:  +0.1389
    Equalized Odds:
      Baseline:     0.3208
      Fair Adapter: 0.1887 ✗
      Improvement:  +0.1321
  AGE:
    Demographic Parity:
      Baseline:     0.0806
      Fair Adapter: 0.0017 ✓
      Improvement:  +0.0789
    Equalized Odds:
      Baseline:     0.1149
      Fair Adapter: 0.0323 ✗
      Improvement:  +0.0827

  Max DP = 0.0222 (⚠ Target not met)
  Max EO = 0.1887 (⚠ Target not met)

▶ PROCESSING BANK DATASET
Cached Bank data to /kaggle/working/preproc_bank.pkl


...


BANK Results:
--------------------------------------------------------------------------------
Baseline Accuracy:       0.8445
Fair Adapter Accuracy:   0.7744
Accuracy Change:         -0.0701

Fairness Metrics:
  MARITAL:
    Demographic Parity:
      Baseline:     0.0788
      Fair Adapter: 0.0029 ✓
      Improvement:  +0.0759
    Equalized Odds:
      Baseline:     0.0632
      Fair Adapter: 0.0228 ✗
      Improvement:  +0.0404
  JOB:
    Demographic Parity:
      Baseline:     0.3650
      Fair Adapter: 0.0114 ✗
      Improvement:  +0.3536
    Equalized Odds:
      Baseline:     0.4667
      Fair Adapter: 0.0333 ✗
      Improvement:  +0.4333

  Max DP = 0.0114 (⚠ Target not met)
  Max EO = 0.0333 (⚠ Target not met)

▶ PROCESSING LAWSCHOOL DATASET
Cached Law School data to /kaggle/working/preproc_lawschool.pkl


INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'pred': 'N', 'race': 'N', 'gender': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'pred': 'N', 'race': 'N', 'gender': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'pred': 'N', 'race': 'N', 'gender': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'pred': 'N', 'race': 'N', 'gender': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'pred': 'N', 'race': 'N', 'gender': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'pred': 'N', 'race': 'N', 'gender': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'pred': 'N',


LAWSCHOOL Results:
--------------------------------------------------------------------------------
Baseline Accuracy:       1.0000
Fair Adapter Accuracy:   0.9192
Accuracy Change:         -0.0808

Fairness Metrics:
  RACE:
    Demographic Parity:
      Baseline:     0.1000
      Fair Adapter: 0.0003 ✓
      Improvement:  +0.0997
    Equalized Odds:
      Baseline:     1.0000
      Fair Adapter: 0.0030 ✓
      Improvement:  +0.9970
  GENDER:
    Demographic Parity:
      Baseline:     0.0188
      Fair Adapter: 0.0005 ✓
      Improvement:  +0.0183
    Equalized Odds:
      Baseline:     0.0000
      Fair Adapter: 0.0075 ✓
      Improvement:  -0.0075

  Max DP = 0.0005 (✓✓✓ SUCCESS)
  Max EO = 0.0075 (✓✓✓ SUCCESS)

▶ PROCESSING HOSPITAL DATASET
Cached Hospital data to /kaggle/working/preproc_hospital.pkl


INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'pred': 'N', 'race': 'N', 'sex': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'pred': 'N', 'race': 'N', 'sex': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'pred': 'N', 'race': 'N', 'sex': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'pred': 'N', 'race': 'N', 'sex': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'pred': 'N', 'race': 'N', 'sex': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'pred': 'N', 'race': 'N', 'sex': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'pred': 'N', 'race': 'N', 'sex


HOSPITAL Results:
--------------------------------------------------------------------------------
Baseline Accuracy:       0.6288
Fair Adapter Accuracy:   0.5636
Accuracy Change:         -0.0652

Fairness Metrics:
  SEX:
    Demographic Parity:
      Baseline:     0.0055
      Fair Adapter: 0.0026 ✓
      Improvement:  +0.0029
    Equalized Odds:
      Baseline:     0.0196
      Fair Adapter: 0.0059 ✓
      Improvement:  +0.0137
  RACE:
    Demographic Parity:
      Baseline:     0.2677
      Fair Adapter: 0.0066 ✓
      Improvement:  +0.2611
    Equalized Odds:
      Baseline:     0.3570
      Fair Adapter: 0.0121 ✗
      Improvement:  +0.3449

  Max DP = 0.0066 (✓✓✓ SUCCESS)
  Max EO = 0.0121 (⚠ Target not met)

FINAL SUMMARY

COMPAS:
  Accuracy: 0.6980 → 0.6372 (-0.0607)
  Max DP: 0.2106 ✗
  Max EO: 1.0000 ✗

GERMAN:
  Accuracy: 0.7250 → 0.7300 (+0.0050)
  Max DP: 0.0222 ✗
  Max EO: 0.1887 ✗

BANK:
  Accuracy: 0.8445 → 0.7744 (-0.0701)
  Max DP: 0.0114 ✗
  Max EO: 0.0333 ✗

LAWSCH